# Semantic simialrity
running in a separate notebook to work on two tasks at once. decided to use gemma

In [1]:
import ollama
import json
import numpy as np
import random
import re
import time
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 317.35it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def generate_questions(chunk_text, model="gemma3:1b", num_questions=3):
    """Generate questions that the chunk should be able to answer."""
    prompt = (
        
        f"Based on the following scientific text, generate exactly {num_questions} "
        "questions that can be answered using ONLY this text. "
        "Return ONLY a JSON list of strings. No explanation, no markdown.\n\n"
        f"Text: {chunk_text}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.7},
    )
    raw = response["message"]["content"].strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if "```" in raw:
            raw = raw.rsplit("```", 1)[0]
    raw = raw.strip()
    
    questions = json.loads(raw)
    if not isinstance(questions, list):
        raise ValueError(f"Expected list, got {type(questions)}")
    return [str(q) for q in questions]

test_path = DATA_DIR / "chunks" / "recursive_1000" / "algae-2020-35-5-31.json"
with open(test_path, encoding="utf-8") as f:
    doc = json.load(f)

chunk = doc["chunks"][0]
questions = generate_questions(chunk["text"])
print("Questions:")
for q in questions:
    print(f"  {q}")

results for gemma 1b:
  What is the name of the species being investigated?
  Where was the species discovered?
  What is the primary focus of the study?

In [7]:
def answer_question(question, context, model="gemma3:1b"):
    """Answer a question given some context."""
    prompt = (
        "Answer the following question based ONLY on the provided context. "
        "Keep your answer concise, 1-3 sentences.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0},
    )
    return response["message"]["content"].strip()


def get_neighbor_chunks(chunk_text, collection, doc_filename, k=3):
    """Retrieve top-k most similar chunks from the same document, excluding the chunk itself."""
    query_embedding = embed_model.embed_query(chunk_text)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k + 1,
        where={"filename": doc_filename}
    )
    neighbors = [
        doc for doc, dist in zip(results["documents"][0], results["distances"][0])
        if dist > 0.01
    ]
    return neighbors[:k]


def cosine_sim(text_a, text_b):
    """Cosine similarity between two texts."""
    vec_a = np.array(embed_model.embed_query(text_a))
    vec_b = np.array(embed_model.embed_query(text_b))
    return float(np.dot(vec_a, vec_b) / (np.linalg.norm(vec_a) * np.linalg.norm(vec_b)))


def ζ_sem_chunk(chunk, collection, doc_filename, model="gemma3:1b"):
    """Compute semantic independence for a single chunk."""
    questions = generate_questions(chunk["text"], model=model)
    neighbors = get_neighbor_chunks(chunk["text"], collection, doc_filename)

    if not neighbors:
        return np.nan

    neighbor_context = "\n\n".join(neighbors)

    similarities = []
    for q in questions:
        answer_alone = answer_question(q, chunk["text"], model=model)
        answer_with_context = answer_question(q, chunk["text"] + "\n\n" + neighbor_context, model=model)
        sim = cosine_sim(answer_alone, answer_with_context)
        similarities.append(max(sim, 0.0))

    return float(np.mean(similarities))


def ζ_sem(chunks, collection, doc_filename, sample_n=5, model="gemma3:1b"):
    """Compute ζ_sem for a document's chunks."""
    if len(chunks) > sample_n:
        sampled = random.sample(chunks, sample_n)
    else:
        sampled = chunks

    scores = []
    failed = 0
    for chunk in sampled:
        try:
            score = ζ_sem_chunk(chunk, collection, doc_filename, model=model)
            if not np.isnan(score):
                scores.append({"chunk_id": chunk["chunk_id"], "score": score})
        except (json.JSONDecodeError, ValueError, KeyError) as e:
            failed += 1
            print(f"  Chunk {chunk['chunk_id']} failed: {e}")

    avg = float(np.mean([s["score"] for s in scores])) if scores else 0.0
    return {"avg": avg, "n_evaluated": len(scores), "n_failed": failed, "chunks": scores}

In [8]:
random.seed(420)

# Reuse sampled docs from ζ_con, take first 15 for bootstrap
with open(PROJECT_ROOT / "outputs" / "HOPE" / "hope_con_results.json", encoding="utf-8") as f:
    ζ_con_raw = json.load(f)
sampled_docs = sorted({key.split("/")[1] for key in ζ_con_raw.keys()})[:15]
print(f"Using {len(sampled_docs)} docs")

chroma_client = chromadb.PersistentClient(path=str(DATA_DIR / "chromadb"))
collections = {
    "recursive_1000": chroma_client.get_collection(name="recursive_100"),
    "semantic_p95": chroma_client.get_collection(name="semantic_p95"),
    "rsc": chroma_client.get_collection(name="rsc"),
}
chunk_dirs = {
    "recursive_1000": DATA_DIR / "chunks" / "recursive_1000",
    "semantic_p95": DATA_DIR / "chunks" / "semantic_p95",
    "rsc": DATA_DIR / "chunks" / "rsc",
}

ζ_sem_path = PROJECT_ROOT / "outputs" / "HOPE" / "hope_sem_results.json"
ζ_sem_path.parent.mkdir(parents=True, exist_ok=True)

if ζ_sem_path.exists():
    with open(ζ_sem_path, encoding="utf-8") as f:
        ζ_sem_results = json.load(f)
    print(f"Resuming — {len(ζ_sem_results)} entries done")
else:
    ζ_sem_results = {}

total = len(sampled_docs) * len(collections)
done = sum(1 for key in ζ_sem_results if any(key.startswith(s + "/") for s in collections))

for doc_name in sampled_docs:
    for strat_name in collections:
        key = f"{strat_name}/{doc_name}"
        if key in ζ_sem_results:
            done += 1
            continue

        chunk_path = chunk_dirs[strat_name] / doc_name
        with open(chunk_path, encoding="utf-8") as f:
            doc = json.load(f)

        print(f"[{done+1}/{total}] {key} ({doc['num_chunks']} chunks)")
        start = time.time()

        result = ζ_sem(doc["chunks"], collections[strat_name], doc["filename"])

        elapsed = time.time() - start
        result["elapsed_s"] = round(elapsed, 1)
        print(f"  ζ_sem={result['avg']:.4f} | {result['n_evaluated']} ok, {result['n_failed']} failed | {elapsed:.0f}s")

        ζ_sem_results[key] = result
        with open(ζ_sem_path, "w", encoding="utf-8") as f:
            json.dump(ζ_sem_results, f, indent=2, ensure_ascii=False)
        done += 1

print("\nDone! Results at:", ζ_sem_path)

Using 15 docs
[1/45] recursive_1000/algae-1986-1-1-241.json (20 chunks)
  ζ_sem=0.7943 | 5 ok, 0 failed | 792s
[2/45] semantic_p95/algae-1986-1-1-241.json (9 chunks)
  Chunk 0 failed: Expecting value: line 1 column 1 (char 0)
  ζ_sem=0.6838 | 4 ok, 1 failed | 1058s
[3/45] rsc/algae-1986-1-1-241.json (18 chunks)
  ζ_sem=0.8100 | 5 ok, 0 failed | 876s
[4/45] recursive_1000/algae-1992-7-1-147.json (31 chunks)
  ζ_sem=0.8502 | 5 ok, 0 failed | 460s
[5/45] semantic_p95/algae-1992-7-1-147.json (9 chunks)
  ζ_sem=0.7986 | 5 ok, 0 failed | 1289s
[6/45] rsc/algae-1992-7-1-147.json (22 chunks)
  Chunk 3 failed: Extra data: line 4 column 1 (char 103)
  ζ_sem=0.8734 | 4 ok, 1 failed | 468s
[7/45] recursive_1000/algae-1997-12-4-269.json (26 chunks)
  ζ_sem=0.8415 | 5 ok, 0 failed | 642s
[8/45] semantic_p95/algae-1997-12-4-269.json (22 chunks)
  Chunk 0 failed: Expecting value: line 1 column 1 (char 0)
  ζ_sem=0.7628 | 4 ok, 1 failed | 1187s
[9/45] rsc/algae-1997-12-4-269.json (26 chunks)
  ζ_sem=0.